# Per-User Profile Generation (structured + LLM bios)

For each unique participant (`iid`) in the Speed Dating data, build a structured JSON profile
(demographics, top hobbies, partner-preference weights, self-perception) and two short
LLM-written paragraphs (`bio` + `seeking`) generated locally via Ollama.

Output: one combined `profiles_<model>.json` keyed by `iid`. The model name is a config knob so
different local models can be compared without overwriting each other's output.

**Workflow:** set `SAMPLE_N = 3` for a fast smoke test, then `SAMPLE_N = None` for the full run.

In [1]:
# --- Config ---
MODEL = "ministral-3:8b"          # swap to compare models, e.g. "qwen2.5:7b-instruct", "gemma3:4b"
OLLAMA_URL = "http://localhost:11434/api/chat"
SAMPLE_N = None                    # set to None for the full 523-user run
CSV_PATH = "dataset/speed_dating_clean.csv"

import re, json
model_slug = re.sub(r"[^0-9a-zA-Z]+", "_", MODEL).strip("_")
OUT_PATH = f"profiles_{model_slug}.json"
print("model:", MODEL, "| output:", OUT_PATH, "| sample:", SAMPLE_N)

model: ministral-3:8b | output: profiles_ministral_3_8b.json | sample: None


In [2]:
# --- Load + dedupe to one row per person ---
import pandas as pd

df = pd.read_csv(CSV_PATH).drop_duplicates("iid").reset_index(drop=True)
print("unique users:", len(df))
df[["iid", "age", "gender", "field_cd", "career_c"]].head()

unique users: 523


,iid,age,gender,field_cd,career_c
0,1,21.0,0,1.0,NaN
1,2,24.0,0,1.0,NaN
2,3,25.0,0,2.0,NaN
3,4,23.0,0,1.0,1.0
4,5,21.0,0,1.0,1.0


In [3]:
# --- Decoder maps (from Speed Dating Data Key.md) ---
GENDER = {0: "Female", 1: "Male"}

FIELD = {
    1: "Law", 2: "Math", 3: "Social Science / Psychology",
    4: "Medical / Pharma / Bio Tech", 5: "Engineering",
    6: "English / Creative Writing / Journalism", 7: "History / Religion / Philosophy",
    8: "Business / Econ / Finance", 9: "Education / Academia",
    10: "Biological Sciences / Chemistry / Physics", 11: "Social Work",
    12: "Undergrad / undecided", 13: "Political Science / International Affairs",
    14: "Film", 15: "Fine Arts / Arts Administration", 16: "Languages",
    17: "Architecture", 18: "Other",
}

CAREER = {
    1: "Lawyer", 2: "Academic / Research", 3: "Psychologist", 4: "Doctor / Medicine",
    5: "Engineer", 6: "Creative Arts / Entertainment",
    7: "Banking / Consulting / Finance / Business", 8: "Real Estate",
    9: "International / Humanitarian Affairs", 10: "Undecided", 11: "Social Work",
    12: "Speech Pathology", 13: "Politics", 14: "Pro sports / Athletics",
    15: "Other", 16: "Journalism", 17: "Architecture",
}

RACE = {
    1: "Black / African American", 2: "European / Caucasian-American",
    3: "Latino / Hispanic American", 4: "Asian / Pacific Islander / Asian-American",
    5: "Native American", 6: "Other",
}

GOAL = {
    1: "a fun night out", 2: "to meet new people", 3: "to get a date",
    4: "looking for a serious relationship", 5: "to say I did it", 6: "other",
}

FREQ = {
    1: "several times a week", 2: "twice a week", 3: "once a week",
    4: "twice a month", 5: "once a month", 6: "several times a year", 7: "almost never",
}

def decode(mapping, value):
    """Look up a coded value, returning 'Unknown' for NaN / out-of-range codes."""
    if pd.isna(value):
        return "Unknown"
    return mapping.get(int(value), "Unknown")

def num(value):
    """Return a plain number (rounded) or None for NaN, for JSON-friendly output."""
    if pd.isna(value):
        return None
    return round(float(value), 2)

def sentiment(score):
    """Map a 1-10 interest score to a sentiment label (None if missing). Float-safe bins."""
    if score is None:
        return None
    if score >= 8:
        return "love it"
    if score >= 6:
        return "cool with it"
    if score >= 5:
        return "meh"
    if score >= 3:
        return "don't like it"
    return "hate it"

SENTIMENT_KEY = {  # label -> by_sentiment bucket name
    "love it": "love_it", "cool with it": "cool_with", "meh": "meh",
    "don't like it": "dislike", "hate it": "hate",
}

HOBBIES = [
    "sports", "tvsports", "exercise", "dining", "museums", "art", "hiking", "gaming",
    "clubbing", "reading", "tv", "theater", "movies", "concerts", "music", "shopping", "yoga",
]
HOBBY_LABEL = {
    "sports": "playing sports", "tvsports": "watching sports", "exercise": "exercising",
    "dining": "dining out", "museums": "museums & galleries", "art": "art",
    "hiking": "hiking & camping", "gaming": "gaming", "clubbing": "dancing & clubbing",
    "reading": "reading", "tv": "watching TV", "theater": "theater", "movies": "movies",
    "concerts": "concerts", "music": "music", "shopping": "shopping", "yoga": "yoga & meditation",
}

# 6 partner-preference traits -> (normalized weight col, self-rating col, label)
TRAITS = [
    ("attractive",       "attr1_1_norm",  "attr3_1"),
    ("sincere",          "sinc1_1_norm",  "sinc3_1"),
    ("intelligent",      "intel1_1_norm", "intel3_1"),
    ("fun",              "fun1_1_norm",   "fun3_1"),
    ("ambitious",        "amb1_1_norm",   "amb3_1"),
    ("shared_interests", "shar1_1_norm",  None),
]
print("decoders ready")

decoders ready


In [4]:
# --- Build the deterministic structured profile block ---
def build_structured_profile(row):
    # hobby interest scores (1-10) -> per-hobby sentiment + grouped-by-sentiment buckets
    scores = {h: num(row[h]) for h in HOBBIES}
    hobby_sentiment = {}
    by_sentiment = {"love_it": [], "cool_with": [], "meh": [], "dislike": [], "hate": []}
    for h in HOBBIES:
        label = sentiment(scores[h])
        if label is None:  # missing score -> skip
            continue
        hobby_sentiment[h] = label
        by_sentiment[SENTIMENT_KEY[label]].append(h)

    # partner-preference weights (normalized, cross-scale comparable) + most valued
    weights = {name: num(row[wcol]) for name, wcol, _ in TRAITS}
    rated_w = [(name, weights[name]) for name in weights if weights[name] is not None]
    most_valued = [n for n, _ in sorted(rated_w, key=lambda x: x[1], reverse=True)[:3]]

    self_perception = {name: num(row[scol]) for name, _, scol in TRAITS if scol}

    return {
        "user_id": int(row["iid"]),
        "demographics": {
            "age": num(row["age"]),
            "gender": decode(GENDER, row["gender"]),
            "race": decode(RACE, row["race"]),
            "field_of_study": decode(FIELD, row["field_cd"]),
            "career": decode(CAREER, row["career_c"]),
        },
        "dating_context": {
            "goal": decode(GOAL, row["goal"]),
            "dates_frequency": decode(FREQ, row["date"]),
            "goes_out": decode(FREQ, row["go_out"]),
            "expected_happiness": num(row["exphappy"]),
        },
        "partner_importance": {
            "same_race": num(row["imprace"]),
            "same_religion": num(row["imprelig"]),
        },
        "interests": {
            "scores": scores,
            "sentiment": hobby_sentiment,
            "by_sentiment": by_sentiment,
        },
        "partner_preferences": {"weights": weights, "most_valued": most_valued},
        "self_perception": self_perception,
    }

# quick look
import json as _json
print(_json.dumps(build_structured_profile(df.iloc[0]), indent=2))

{
  "user_id": 1,
  "demographics": {
    "age": 21.0,
    "gender": "Female",
    "race": "Asian / Pacific Islander / Asian-American",
    "field_of_study": "Law",
    "career": "Unknown"
  },
  "dating_context": {
    "goal": "to meet new people",
    "dates_frequency": "almost never",
    "goes_out": "several times a week",
    "expected_happiness": 3.0
  },
  "partner_importance": {
    "same_race": 2.0,
    "same_religion": 4.0
  },
  "interests": {
    "scores": {
      "sports": 9.0,
      "tvsports": 2.0,
      "exercise": 8.0,
      "dining": 9.0,
      "museums": 1.0,
      "art": 1.0,
      "hiking": 5.0,
      "gaming": 1.0,
      "clubbing": 5.0,
      "reading": 6.0,
      "tv": 9.0,
      "theater": 1.0,
      "movies": 10.0,
      "concerts": 10.0,
      "music": 9.0,
      "shopping": 8.0,
      "yoga": 1.0
    },
    "sentiment": {
      "sports": "love it",
      "tvsports": "hate it",
      "exercise": "love it",
      "dining": "love it",
      "museums": "hate it"

In [5]:
# --- Render a structured profile as a readable brief for the LLM ---
def _names(keys):
    return ", ".join(HOBBY_LABEL.get(h, h) for h in keys)

def profile_to_prompt(p):
    d, ctx = p["demographics"], p["dating_context"]
    bys = p["interests"]["by_sentiment"]
    age = d["age"] if d["age"] is not None else "unknown-age"
    loves = _names(bys["love_it"]) or "nothing in particular"
    enjoys = _names(bys["cool_with"])
    dislikes = _names(bys["dislike"] + bys["hate"])
    valued = ", ".join(t.replace("_", " ") for t in p["partner_preferences"]["most_valued"]) or "a good connection"
    imp = p["partner_importance"]
    lines = [
        f"Age: {age}",
        f"Gender: {d['gender']}",
        f"Race/ethnicity: {d['race']}",
        f"Field of study: {d['field_of_study']}",
        f"Career: {d['career']}",
        f"Reason for joining: {ctx['goal']}",
        f"Goes out: {ctx['goes_out']}; dates: {ctx['dates_frequency']}",
        f"Loves: {loves}",
    ]
    if enjoys:
        lines.append(f"Enjoys: {enjoys}")
    if dislikes:
        lines.append(f"Not into: {dislikes}")
    lines.append(f"Most values in a partner: {valued}")
    lines.append(
        f"Importance of same race (1-10): {imp['same_race']}; same religion (1-10): {imp['same_religion']}"
    )
    return "\n".join(lines)

print(profile_to_prompt(build_structured_profile(df.iloc[0])))

Age: 21.0
Gender: Female
Race/ethnicity: Asian / Pacific Islander / Asian-American
Field of study: Law
Career: Unknown
Reason for joining: to meet new people
Goes out: several times a week; dates: almost never
Loves: playing sports, exercising, dining out, watching TV, movies, concerts, music, shopping
Enjoys: reading
Not into: watching sports, museums & galleries, art, gaming, theater, yoga & meditation
Most values in a partner: sincere, intelligent, attractive
Importance of same race (1-10): 2.0; same religion (1-10): 4.0


In [6]:
# --- Ollama call (stdlib urllib; .venv has no requests) ---
import urllib.request

SYSTEM_PROMPT = (
    "You write dating-app profiles. Given a participant's structured details, write two short, "
    "warm, first-person paragraphs. Respond with ONLY a JSON object with exactly two keys: "
    "'bio' (2-3 sentences about who they are, their studies/career and what they love doing; if "
    "they have a clear 'Not into' interest, you may work in one light dislike naturally, e.g. "
    "\"though you won't catch me clubbing\") and 'seeking' (1-2 sentences about what they want in a "
    "partner, centered on the traits they value; keep it inviting, not a list of dealbreakers). "
    "Be natural and specific to the details; do not invent facts not present; no markdown. "
    "Do NOT use a name or any placeholder like '[Name]' — write anonymously in the first person."
)

def call_ollama(prompt, model=MODEL, timeout=120):
    payload = {
        "model": model,
        "stream": False,
        "format": "json",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
    }
    req = urllib.request.Request(
        OLLAMA_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        content = json.loads(resp.read())["message"]["content"]
    try:
        obj = json.loads(content)
    except json.JSONDecodeError:
        return {"bio": content.strip(), "seeking": ""}
    return {"bio": str(obj.get("bio", "")).strip(),
            "seeking": str(obj.get("seeking", "")).strip()}

# smoke-test the call on user 0
call_ollama(profile_to_prompt(build_structured_profile(df.iloc[0])))

{'bio': 'I’m currently studying law and trying to figure out how I’ll actually use it all someday—though for now, I’d rather be out on the court playing basketball or hitting up a new restaurant with friends than stuck in a library. When I’m not moving around (or eating), you can find me curled up with a book, jamming to music, or planning my next shopping haul. Though you won’t catch me at a concert unless it’s for something like Taylor Swift—my taste runs more indie and nostalgic than stadium rock.',
 'seeking': 'I’m looking for someone who sees the world with curiosity and doesn’t take themselves too seriously, even if they’re brilliant about it. A partner who appreciates my energy but also knows when to slow down for a quiet chat or a shared laugh feels like the perfect balance—no pressure, just good vibes.'}

In [7]:
# --- Generate loop ---
rows = df if SAMPLE_N is None else df.head(SAMPLE_N)
profiles = {}
failures = []

for i, (_, row) in enumerate(rows.iterrows(), 1):
    prof = build_structured_profile(row)
    try:
        text = call_ollama(profile_to_prompt(prof))
    except Exception as e:
        text = {"bio": "", "seeking": ""}
        failures.append((prof["user_id"], repr(e)))
    prof["bio"] = text["bio"]
    prof["seeking"] = text["seeking"]
    profiles[str(prof["user_id"])] = prof
    if i % 10 == 0 or i == len(rows):
        print(f"  {i}/{len(rows)} done")

print("generated:", len(profiles), "| failures:", len(failures))
if failures:
    print(failures[:5])

  10/523 done
  20/523 done
  30/523 done
  40/523 done
  50/523 done
  60/523 done
  70/523 done
  80/523 done
  90/523 done
  100/523 done
  110/523 done
  120/523 done
  130/523 done
  140/523 done
  150/523 done
  160/523 done
  170/523 done
  180/523 done
  190/523 done
  200/523 done
  210/523 done
  220/523 done
  230/523 done
  240/523 done
  250/523 done
  260/523 done
  270/523 done
  280/523 done
  290/523 done
  300/523 done
  310/523 done
  320/523 done
  330/523 done
  340/523 done
  350/523 done
  360/523 done
  370/523 done
  380/523 done
  390/523 done
  400/523 done
  410/523 done
  420/523 done
  430/523 done
  440/523 done
  450/523 done
  460/523 done
  470/523 done
  480/523 done
  490/523 done
  500/523 done
  510/523 done
  520/523 done
  523/523 done
generated: 523 | failures: 0


In [8]:
# --- Write + preview ---
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(profiles, f, indent=2, ensure_ascii=False)
print("wrote", len(profiles), "profiles ->", OUT_PATH)

sample_key = next(iter(profiles))
print(json.dumps(profiles[sample_key], indent=2, ensure_ascii=False))

wrote 523 profiles -> profiles_ministral_3_8b.json
{
  "user_id": 1,
  "demographics": {
    "age": 21.0,
    "gender": "Female",
    "race": "Asian / Pacific Islander / Asian-American",
    "field_of_study": "Law",
    "career": "Unknown"
  },
  "dating_context": {
    "goal": "to meet new people",
    "dates_frequency": "almost never",
    "goes_out": "several times a week",
    "expected_happiness": 3.0
  },
  "partner_importance": {
    "same_race": 2.0,
    "same_religion": 4.0
  },
  "interests": {
    "scores": {
      "sports": 9.0,
      "tvsports": 2.0,
      "exercise": 8.0,
      "dining": 9.0,
      "museums": 1.0,
      "art": 1.0,
      "hiking": 5.0,
      "gaming": 1.0,
      "clubbing": 5.0,
      "reading": 6.0,
      "tv": 9.0,
      "theater": 1.0,
      "movies": 10.0,
      "concerts": 10.0,
      "music": 9.0,
      "shopping": 8.0,
      "yoga": 1.0
    },
    "sentiment": {
      "sports": "love it",
      "tvsports": "hate it",
      "exercise": "love it",
  